# Social Media Addiction vs Productivity - EDA & Classification

**Dataset:** [Social Media Addiction vs Productivity](https://www.kaggle.com/datasets/asifxzaman/social-media-addiction-vs-productivity-dataset)

## Overview
This notebook explores the relationship between social media usage and productivity across 6000 individuals. We run statistical tests to identify which factors drive addiction severity, then build a Random Forest classifier to predict addiction level from behavioral features.

**Sections:**
1. Data Loading
2. Data Cleaning
3. Exploratory Data Analysis
4. Normality Analysis
5. Statistical Analysis
6. ML Preprocessing & Correlation
7. Classification Model
8. Key Takeaways

## 1. Data Loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import kruskal
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

sns.set_style("whitegrid")

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/asifxzaman/social-media-addiction-vs-productivity-dataset/social_media_productivity_6000.csv")
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 2. Data Cleaning

### 2.1 Missing Values

In [ ]:
(df.isna().sum() / len(df)) * 100

Since missing data is only around 2%, the best choice is to fill missing numeric values with the median and categorical values with the mode.

In [ ]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(exclude='number').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum()

## 3. Exploratory Data Analysis

### 3.1 Addiction Level Distribution

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="addiction_level", order=["Low", "Medium", "High"])
plt.title("Addiction Level Distribution")
plt.tight_layout()
plt.show()

### 3.2 Numeric Feature Distributions

In [ ]:
df[num_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

## 4. Normality Analysis

### 4.1 Numeric Distributions

Most numeric features are not normally distributed. Social media hours and focus score are skewed. Productivity score is the closest to normal but has a spike around 0. Since we filled with the median, non-normality is not a concern for imputation.

### 4.2 Q-Q Plots

In [ ]:
for col in num_cols:
    plt.figure(figsize=(5, 4))
    stats.probplot(df[col].dropna(), dist="norm", plot=plt)
    plt.title(f"Q-Q Plot: {col}")
    plt.show()

All features show an S-shape or its derivatives confirming non-normality. We use Kruskal-Wallis for all group comparisons going forward.

## 5. Statistical Analysis

First thing that comes to mind when talking about social media addiction is age, so let's check if age shows statistical significance across addiction levels.

In [ ]:
sns.boxplot(x="addiction_level", y="age", data=df, order=["Low", "Medium", "High"])
plt.title("Age vs Addiction Level")
plt.show()

In [ ]:
def kruskal_test(feature):
    groups = [
        df[df['addiction_level'] == level][feature].dropna()
        for level in ["Low", "Medium", "High"]
    ]
    stat, p = kruskal(*groups)
    result = "Reject null - significant difference" if p < 0.05 else "Fail to reject null - no significant difference"
    print(f"{feature}: H={stat:.2f}, p={p:.4f} -> {result}")

for col in num_cols:
    kruskal_test(col)

Age distributions are nearly identical across addiction levels. We fail to reject the null hypothesis - age does not significantly differ across groups.

In [ ]:
sns.boxplot(x="addiction_level", y="social_media_hours", data=df, order=["Low", "Medium", "High"])
plt.title("Social Media Hours vs Addiction Level")
plt.show()

Social media hours show a clear and statistically significant difference across addiction levels. Higher addiction severity is associated with more daily usage.

In [ ]:
sns.boxplot(x="addiction_level", y="sleep_hours", data=df, order=["Low", "Medium", "High"])
plt.title("Sleep Hours vs Addiction Level")
plt.show()

Sleep hours show no meaningful difference across addiction levels.

In [ ]:
sns.boxplot(x="addiction_level", y="notifications_per_day", data=df, order=["Low", "Medium", "High"])
plt.title("Notifications per Day vs Addiction Level")
plt.show()

Same goes for notifications per day - no significant difference across groups.

## 6. ML Preprocessing & Correlation

In [ ]:
df["addiction_level"] = OrdinalEncoder(categories=[["Low", "Medium", "High"]]).fit_transform(df[["addiction_level"]])

corr = df.corr(numeric_only=True)

corr["addiction_level"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(corr, cmap="coolwarm", annot=True)
plt.title("Feature Correlation Heatmap")
plt.show()

Social media hours has the strongest positive correlation with addiction level, consistent with the statistical test results.

## 7. Classification Model

We predict addiction level using a Random Forest classifier with stratified train/test split and 5-fold cross validation.

### 7.1 Cross Validation

In [ ]:
X = df.drop("addiction_level", axis=1)
y = df["addiction_level"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(random_state=42)

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

print("CV Fold Scores:", cv_scores)
print("CV Mean Accuracy:", round(cv_scores.mean(), 4))
print("CV Std:", round(cv_scores.std(), 4))

### 7.2 Test Set Evaluation

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))

### 7.3 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "Medium", "High"],
            yticklabels=["Low", "Medium", "High"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

### 7.4 Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(8, 5))
importances.plot(kind='barh')
plt.title("Feature Importance - Random Forest")
plt.tight_layout()
plt.show()

Social media hours is the most important feature by a clear margin, consistent with the statistical analysis and correlation results.

## 8. Key Takeaways

| Finding | Implication |
|---|---|
| Social media hours is the only feature with a significant Kruskal-Wallis result | The clearest behavioral driver of addiction severity |
| Age shows no significant difference across addiction levels | Not a useful predictor in this dataset |
| Sleep hours and notifications show no significant group differences | Weaker signals than expected |
| Social media hours has the highest correlation with addiction level | Consistent across all analysis methods |
| Social media hours is the top feature by importance | Statistical tests, correlation, and model all agree |

> This dataset may be synthetically generated given how cleanly balanced the distributions are. Results should be interpreted with that in mind.